In [1]:
import random
from collections import deque, namedtuple
from datetime import datetime

import gymnasium as gym
import torch
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter

from src.agents.common import calc_advantage
from src.networks.dqn import DQN
from src.networks.mlp import MLPBox



In [2]:
LR = 2.5e-3
num_episodes = 3000
K_EPOCH = 4
REPEAT = 1
num_tests=1000
CRITIC_BATCH_SIZE = 32
CRITIC_TTL = 32
GAMMA = .95
EPS = .2
ENTROPY_COEFF = .01

In [3]:
run_name = f"pendulum_ppo_lr{LR}_ne{num_episodes}_k{K_EPOCH}_r{REPEAT}_{datetime.now():%Y%m%d_%H%M%S}"
writer = SummaryWriter(f"./logs/{run_name}")

In [4]:
env = gym.make("Pendulum-v1", render_mode=None)
# agent = PPOCartAgent(env=env, learning_rate=LR)

env.action_space, env.observation_space

(Box(-2.0, 2.0, (1,), float32), Box([-1. -1. -8.], [1. 1. 8.], (3,), float32))

# Init networks

In [5]:
policy = MLPBox(env.observation_space.shape[0], env.action_space.shape[0])
critic = DQN(env.observation_space.shape[0], 1, 20)

optimizer_policy = torch.optim.Adam(policy.net.parameters(), LR)
optimizer_critic = torch.optim.Adam(critic.net.parameters(), LR)

# API

In [6]:
EpStep = namedtuple('episode_step', (
    'state',
    'next_state',
    'action',
    'reward',
    'done',
    'probs',
    ))

In [7]:
@torch.no_grad
def get_critiques(critic: torch.nn.Module, observations: torch.Tensor): 
    return critic(observations).squeeze(-1)

def evaluate_action(policy: torch.nn.Module, observations: torch.Tensor, actions: torch.Tensor):
    mean, std = policy(observations)
    dist = torch.distributions.Normal(mean, std)
    dist.log_prob(actions)
    return dist.log_prob(actions), dist.entropy().mean()

def update_critic(
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    batch: list[EpStep],
    gamma=GAMMA,
    ): 
    batch = EpStep(*zip(*episode_history, strict=False))

    states = torch.stack(batch.state) 
    next_states = torch.stack(batch.next_state)
    rewards = torch.stack(batch.reward)
    dones = torch.stack(batch.done)


    q_pred = model(states).squeeze(1)
    with torch.no_grad():
        q_next = model(next_states).squeeze(1)
        q_target = rewards + gamma * q_next * (1 - dones)

    loss = F.mse_loss(q_pred, q_target)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

def update_policy(
    model: torch.nn.Module,
    critic: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    batch: list[EpStep],
    gamma=GAMMA,
):
    batch = EpStep(*zip(*episode_history, strict=False))

    old_log_probs = torch.stack(batch.probs).detach()
    actions_t     = torch.stack(batch.action)
    rewards       = torch.stack(batch.reward)
    states_t      = torch.stack(batch.state) 

    critiques = get_critiques(critic, states_t)

    new_log_probs, entropy = evaluate_action(model, states_t, actions_t)

    ratio = torch.exp(new_log_probs - old_log_probs)

    advantage = calc_advantage(rewards, critiques, gamma)

    surr1 = ratio * advantage
    surr2 = torch.clamp(ratio, 1 - EPS, 1 + EPS) * advantage
    actor_loss = -torch.min(surr1, surr2).mean()
    
    loss = actor_loss - entropy * ENTROPY_COEFF

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()



def get_action(policy: torch.nn.Module, observations: torch.Tensor):
    mean, std = policy(observations)
    dist = torch.distributions.Normal(mean, std)
    action = dist.sample()
    return torch.clamp(action, -2.0, 2.0), action, dist.log_prob(action).sum(-1).detach()


# Training

In [8]:
for e in range(num_episodes):
    state, info = env.reset()
    done = False

    episode_history = deque([], maxlen=10_000)

    step = 0
    while not done:
        state = torch.tensor(state, dtype=torch.float32)
        action, raw_action, prob = get_action(policy, state)
        next_state, reward, terminated, truncated, info = env.step(action.detach().numpy())
        done = terminated or truncated

        if len(episode_history) >= CRITIC_BATCH_SIZE and step % CRITIC_TTL == 0:
            update_critic(
                critic,
                optimizer_critic,
                random.sample(episode_history, CRITIC_BATCH_SIZE))

        episode_history.append(EpStep(
            state,
            torch.tensor(next_state, dtype=torch.float32),
            raw_action,
            torch.tensor(reward, dtype=torch.float32),
            torch.tensor(done, dtype=torch.int8),
            prob))
        state = next_state
        step += 1



    # TRAINING PASS:
    for _ in range(K_EPOCH):
        update_policy(policy, critic, optimizer_policy, episode_history)

    avg_reward = (sum(step.reward for step in episode_history) / len(episode_history)).item()
    writer.add_scalar("train/reward", avg_reward, e + 1)

In [ ]:
dummy_input = torch.randn(1, env.observation_space.shape[0])
torch.onnx.export(
    policy,
    dummy_input,
    f"./data/latest.onnx",
    input_names=["obs"],
    output_names=["action_probs"],
    dynamic_axes={"obs": {0: "batch"}, "action_probs": {0: "batch"}},
    external_data=False,
)
print(f"./data/{run_name}.onnx")